# Phase 1
- make llm calls
- history

In [93]:
import os
import litellm


# Turn 1

history = [{"role": "system", "content": "You're NanoClaude Code, a god tier coding agent"}]

# User message
history.append({"role": "user", "content": "Where is Mount Everest Situated?"})

r1 = litellm.completion(
    model="deepseek/deepseek-v4-pro", # openai/gpt-5.2
    messages=history,
    thinking={"type": "enabled"},
    api_key=os.environ['LLM_API_KEY']
)


msg1 = r1.choices[0].message

print(msg1)


Message(content='Mount Everest is located in the **Himalayan mountain range**, straddling the border between **Nepal** (to the south) and the **Tibet Autonomous Region of China** (to the north). Its summit sits at 8,848.86 metres (29,031.7 ft) above sea level, making it the highest point on Earth.', role='assistant', tool_calls=None, function_call=None, reasoning_content='We need to answer: "Where is Mount Everest Situated?" The user is asking a straightforward factual question. The response should be concise and accurate. Mount Everest is located in the Himalayas, on the border between Nepal and the Tibet Autonomous Region of China. We can mention both countries, its height, and perhaps its coordinates. But "where is it situated" likely expects the country/region. I\'ll provide a clear answer. No need for code or deep analysis.', provider_specific_fields={'reasoning_content': 'We need to answer: "Where is Mount Everest Situated?" The user is asking a straightforward factual question. 

In [3]:
r1.choices[0]

Choices(finish_reason='stop', index=0, message=Message(content='Mount Everest (known as Sagarmāthā in Nepal and Chomolungma in Tibet) is situated in the **Mahalangur Himal sub-range of the Himalayas**, straddling the international border between **Nepal** (Province No. 1) and the **Tibet Autonomous Region of China**. Its summit lies precisely on the boundary line between the two countries.', role='assistant', tool_calls=None, function_call=None, reasoning_content='We need to answer the question: "Where is Mount Everest Situated?" This is a straightforward factual question. The model is NanoClaude Code, a god tier coding agent. The user is asking a simple geography question. I should answer accurately and concisely, but I can also add a bit of context. Since the prompt says "NanoClaude Code, a god tier coding agent", maybe they want a coding-related response? But the question is not about coding. I\'ll just answer the question. Maybe the user is testing the agent\'s general knowledge. I

In [4]:
history.append({"role": "assistant", "content": msg1.content})
history.append({"role": "user", "content": 'and whats the first basecamp there?'})

# Turn 2
r2 = litellm.completion(
    model="deepseek/deepseek-v4-pro",
    messages=history,
    thinking={"type": "enabled"},
    api_key=os.environ['LLM_API_KEY']
)

print("Turn 2 response:", r2.choices[0].message.content)

Turn 2 response: The answer depends on how you define “first” or “main” base camp. There are actually **two base camps** on Everest, each on a different side of the mountain:

---

### 1. South Base Camp (Nepal) – The Most Famous “First Base Camp”
- **Location:** Khumbu region, Nepal, at an elevation of roughly **5,364 m (17,598 ft)**.
- **Why it’s considered the “first” or primary base camp:**  
  - This is the starting point for the standard Southeast Ridge (South Col) route, the most popular and frequently climbed route.  
  - It’s the endpoint of the iconic **Everest Base Camp Trek**, so when people talk about “Everest Base Camp” without qualification, they almost always mean the South Base Camp in Nepal.  
  - Most commercial expeditions and climbing records use this as their camp.

---

### 2. North Base Camp (Tibet) – Historically Earlier, Now the Secondary
- **Location:** Tibet Autonomous Region, China, at about **5,150 m (16,900 ft)**.
- **Why it might be considered “first” hi

# Phase 2

add tools

In [11]:
import os
import litellm

litellm.set_verbose = False

history = [{"role": "system", "content": "You're NanoClaude Code, a god tier coding agent"}]

history.append({"role": "user", "content": "What is the weather in Tokyo?"})

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "If you want to get the weather, use this function",
        "parameters": {
            "type": "object",
            "properties": {"location": {"type": "string"}},
            "required": ["location"]
        }
    }
}]

# Turn 1
r1 = litellm.completion(
    model="deepseek/deepseek-v4-pro",
    messages=history,
    thinking={"type": "enabled"},
    tools=tools,
    api_key=os.environ['LLM_API_KEY']
)
msg1 = r1.choices[0].message
assert msg1.tool_calls, "Expected tool call in turn 1"


In [12]:
r1.choices[0]

Choices(finish_reason='tool_calls', index=0, message=Message(content='', role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"location": "Tokyo"}', name='get_weather'), id='call_00_LLzFSX8wJSA5CJbxEJna7955', type='function')], function_call=None, reasoning_content='The user is asking for the weather in Tokyo. I need to use the get_weather function with the location parameter set to "Tokyo".', provider_specific_fields={'reasoning_content': 'The user is asking for the weather in Tokyo. I need to use the get_weather function with the location parameter set to "Tokyo".'}), provider_specific_fields={})

In [13]:
msg1.tool_calls[0].function

Function(arguments='{"location": "Tokyo"}', name='get_weather')

In [10]:
msg1.model_dump()

{'content': '',
 'role': 'assistant',
 'tool_calls': [{'index': 0,
   'function': {'arguments': '{"location": "Tokyo"}', 'name': 'get_weather'},
   'id': 'call_00_s1HAxkukbVlY4XHb96Sa1888',
   'type': 'function'}],
 'function_call': None,
 'reasoning_content': 'The user is asking for the weather in Tokyo. I should use the get_weather function to fetch this information.',
 'provider_specific_fields': {'reasoning_content': 'The user is asking for the weather in Tokyo. I should use the get_weather function to fetch this information.'}}

In [ ]:
def get_weather_fn(location):
    return str({"weather": "Sunny", "temp": "28C", "location": location})

In [16]:
if msg1.tool_calls and msg1.tool_calls[0].function.name == 'get_weather':
    
    history.append(msg1.model_dump())
    # call some external api
    
    location = json.loads(msg1.tool_calls[0].function.arguments)['location']
    output = get_weather_fn(location)
    
    
    history.append({"role": "tool", "content": output, "tool_call_id": msg1.tool_calls[0].id})
    

In [16]:
import json
json.loads(msg1.tool_calls[0].function.arguments)['location']

'Tokyo'

In [17]:
# Turn 2
r2 = litellm.completion(
    model="deepseek/deepseek-v4-pro",
    messages=history,
    thinking={"type": "enabled", "budget_tokens": 1000},
        api_key=os.environ['LLM_API_KEY']
)
print("Turn 2 response:", r2.choices[0].message.content)
print("Multi-turn live test: OK — no HTTP 400")

Turn 2 response: The weather in Tokyo is **Sunny** with a temperature of **28°C** — a beautiful day! ☀️
Multi-turn live test: OK — no HTTP 400


In [22]:
print(r2)

ModelResponse(id='bf637392-bd59-404e-979c-91b3092612e4', created=1778390413, model='deepseek-v4-pro', object='chat.completion', system_fingerprint='fp_9954b31ca7_prod0820_fp8_kvcache_20260402', choices=[Choices(finish_reason='stop', index=0, message=Message(content='The weather in Tokyo is **Sunny** with a temperature of **28°C** — a beautiful day! ☀️', role='assistant', tool_calls=None, function_call=None, reasoning_content='I have the weather information for Tokyo. Let me present it clearly.', provider_specific_fields={'reasoning_content': 'I have the weather information for Tokyo. Let me present it clearly.'}), provider_specific_fields={})], usage=Usage(completion_tokens=41, prompt_tokens=117, total_tokens=158, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=14, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_t

In [24]:
print(r1)

ModelResponse(id='c806b187-7865-4d5a-a386-881a2eb55e5e', created=1778390391, model='deepseek-v4-pro', object='chat.completion', system_fingerprint='fp_9954b31ca7_prod0820_fp8_kvcache_20260402', choices=[Choices(finish_reason='tool_calls', index=0, message=Message(content='', role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"location": "Tokyo"}', name='get_weather'), id='call_00_85vBSXTBgvHLct1fwvOG1957', type='function')], function_call=None, reasoning_content="The user is asking for the weather in Tokyo. I'll use the get_weather function to fetch this information.", provider_specific_fields={'reasoning_content': "The user is asking for the weather in Tokyo. I'll use the get_weather function to fetch this information."}), provider_specific_fields={})], usage=Usage(completion_tokens=68, prompt_tokens=290, total_tokens=358, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoni

# Simple BashSession

First, install libtmux, `pip install libtmux`

In [18]:
import libtmux
import time

class BashSession:
#     PS1 = CmdOutputMetadata.build_ps1_prompt()
    HISTORY_LIMIT = 10_000

    def __init__(self, work_dir: str, username: str):
        self.work_dir = work_dir
        self.username = username
        self.server = libtmux.Server()
    
    def initialize(self):
        self.session = self.server.new_session(
            session_name='bash_session',
            start_directory=self.work_dir,
            kill_session=True,
            x=1000, y=1000,
        )
        self.session.set_option('history-limit', str(self.HISTORY_LIMIT))

        initial_window = self.session.active_window
        
        
        self.window = self.session.new_window(
            window_name='bash',
            window_shell='/bin/bash',
            start_directory=self.work_dir,
        )
        
        initial_window.kill()
        self.pane = self.window.active_pane
        time.sleep(0.5)
        self._clear_pane()

    def _clear_pane(self) -> None:
        self.pane.send_keys('C-l', enter=False)
        time.sleep(0.1)
        self.pane.cmd('clear-history')

    def _get_pane_content(self) -> str:
        lines = self.pane.cmd('capture-pane', '-J', '-pS', '-').stdout
        return '\n'.join(line.rstrip() for line in lines)

    def execute(self, command: str) -> None:
        self.pane.send_keys(
            f'{command}'
        )
        time.sleep(0.5)
        print(self._get_pane_content())

In [19]:
bash_session = BashSession(work_dir='/Users/cohlem/Projects/Experimentation', username='nanoclaudeagent')

In [20]:
bash_session.initialize()

In [21]:
bash_session.execute('ls -lah')

bash-3.2$ ls -lah
total 88
drwxr-xr-x  44 cohlem  staff   1.4K May 10 17:02 .
drwxr-xr-x  18 cohlem  staff   576B May  1 23:15 ..
-rw-r--r--@  1 cohlem  staff    26K May 10 18:48 .DS_Store
drwxr-xr-x   3 cohlem  staff    96B Jul  6  2023 .chroma
drwxr-xr-x   3 cohlem  staff    96B Jul  6  2023 .ipynb_checkpoints
drwxr-xr-x  23 cohlem  staff   736B Sep  7  2025 Deep Learning
drwxr-xr-x   5 cohlem  staff   160B Mar 12  2025 MI
drwxr-xr-x   3 cohlem  staff    96B Mar 16  2025 ML books
drwxr-xr-x   4 cohlem  staff   128B Sep 18  2025 Papers
drwxr-xr-x  17 cohlem  staff   544B Feb  7  2025 Practice
drwxr-xr-x   5 cohlem  staff   160B Jan 29  2025 Python Books
drwxr-xr-x  11 cohlem  staff   352B Sep  5  2025 RL-GRPO
drwxr-xr-x@ 12 cohlem  staff   384B Aug 16  2025 RL-framework
drwxr-xr-x  13 cohlem  staff   416B Mar 10 22:43 RL2
drwxr-xr-x  18 cohlem  staff   576B Mar 10 22:43 RLPR
drwxr-xr-x  18 cohlem  staff   576B May 10 11:49 TinyOH
drwxr-xr-x   4 cohlem  staff   128B Mar 10 22:44 TinyZe

# ADD Prompt

In [23]:
import re
from pydantic import BaseModel
import json

HARD_TIMEOUT = 300

PS1_BLOCK_BEGIN = '\n###PS1JSON###\n'
PS1_BLOCK_END   = '\n###PS1END###'


class CmdOutputMetadata(BaseModel):
    exit_code: str = "-1"
    pid: int = -1
    username: str | None = None
    hostname: str | None = None
    working_dir: str | None = None
    py_interpreter_path: str | None = None

    @classmethod
    def build_ps1_prompt(cls) -> str:
        json_template = json.dumps(
            {
                'pid': '$!',
                'exit_code': '$?',
                'username': r'\u',
                'hostname': r'\h',
                'working_dir': r'$(pwd)',
                'py_interpreter_path': r'$(which python 2>/dev/null || echo "")',
            },
            indent=2,
        ).replace('"', r'\"')

        return PS1_BLOCK_BEGIN + json_template + PS1_BLOCK_END + '\n'



In [24]:

class BashSession:
    PS1 = CmdOutputMetadata.build_ps1_prompt()
    HISTORY_LIMIT = 10_000

    def __init__(self, work_dir: str, username: str):
        self.work_dir = work_dir
        self.username = username
        self.server = libtmux.Server()

    def initialize(self):
        self.session = self.server.new_session(
            session_name='bash_session',
            start_directory=self.work_dir,
            kill_session=True,
            x=1000, y=1000,
        )
        self.session.set_option('history-limit', str(self.HISTORY_LIMIT))

        initial_window = self.session.active_window
        self.window = self.session.new_window(
            window_name='bash',
            window_shell='/bin/bash',
            start_directory=self.work_dir,
        )
        self.pane = self.window.active_pane
        initial_window.kill()

        self.pane.send_keys(
            f'export PROMPT_COMMAND=\'export PS1="{self.PS1}"\'; export PS2=""'
        )
        time.sleep(0.5)
        self._clear_pane()

    def _clear_pane(self) -> None:
        self.pane.send_keys('C-l', enter=False)
        time.sleep(0.1)
        self.pane.cmd('clear-history')

    def _get_pane_content(self) -> str:
        lines = self.pane.cmd('capture-pane', '-J', '-pS', '-').stdout
        return '\n'.join(line.rstrip() for line in lines)


    def execute(self, command: str) -> None:
        self.pane.send_keys(
            f'{command}'
        )
        time.sleep(0.5)
        return (self._get_pane_content())



In [25]:
bash_session = BashSession(work_dir='/Users/cohlem/Projects/Experimentation', username='nanoclaudeagent')
bash_session.initialize()

In [26]:
bash_session.execute('ls -lah')


###PS1JSON###
{
  "pid": "37675",
  "exit_code": "0",
  "username": "cohlem",
  "hostname": "Manishs-MacBook-Air",
  "working_dir": "/Users/cohlem/Projects/Experimentation",
  "py_interpreter_path": "/Users/cohlem/anaconda3/bin/python"
}
###PS1END###
ls -lah
total 88
drwxr-xr-x  44 cohlem  staff   1.4K May 10 17:02 .
drwxr-xr-x  18 cohlem  staff   576B May  1 23:15 ..
-rw-r--r--@  1 cohlem  staff    26K May 10 18:48 .DS_Store
drwxr-xr-x   3 cohlem  staff    96B Jul  6  2023 .chroma
drwxr-xr-x   3 cohlem  staff    96B Jul  6  2023 .ipynb_checkpoints
drwxr-xr-x  23 cohlem  staff   736B Sep  7  2025 Deep Learning
drwxr-xr-x   5 cohlem  staff   160B Mar 12  2025 MI
drwxr-xr-x   3 cohlem  staff    96B Mar 16  2025 ML books
drwxr-xr-x   4 cohlem  staff   128B Sep 18  2025 Papers
drwxr-xr-x  17 cohlem  staff   544B Feb  7  2025 Practice
drwxr-xr-x   5 cohlem  staff   160B Jan 29  2025 Python Books
drwxr-xr-x  11 cohlem  staff   352B Sep  5  2025 RL-GRPO
drwxr-xr-x@ 12 cohlem  staff   384B Au

In [27]:
bash_session.execute('ls videos')


###PS1JSON###
{
  "pid": "37675",
  "exit_code": "0",
  "username": "cohlem",
  "hostname": "Manishs-MacBook-Air",
  "working_dir": "/Users/cohlem/Projects/Experimentation",
  "py_interpreter_path": "/Users/cohlem/anaconda3/bin/python"
}
###PS1END###
ls -lah
total 88
drwxr-xr-x  44 cohlem  staff   1.4K May 10 17:02 .
drwxr-xr-x  18 cohlem  staff   576B May  1 23:15 ..
-rw-r--r--@  1 cohlem  staff    26K May 10 18:48 .DS_Store
drwxr-xr-x   3 cohlem  staff    96B Jul  6  2023 .chroma
drwxr-xr-x   3 cohlem  staff    96B Jul  6  2023 .ipynb_checkpoints
drwxr-xr-x  23 cohlem  staff   736B Sep  7  2025 Deep Learning
drwxr-xr-x   5 cohlem  staff   160B Mar 12  2025 MI
drwxr-xr-x   3 cohlem  staff    96B Mar 16  2025 ML books
drwxr-xr-x   4 cohlem  staff   128B Sep 18  2025 Papers
drwxr-xr-x  17 cohlem  staff   544B Feb  7  2025 Practice
drwxr-xr-x   5 cohlem  staff   160B Jan 29  2025 Python Books
drwxr-xr-x  11 cohlem  staff   352B Sep  5  2025 RL-GRPO
drwxr-xr-x@ 12 cohlem  staff   384B Au

        <RESULT>ls videos
        comparision     day x of

        ls Papers
        RL

        </RESULT>
        [Current working directory: /Users/cohlem/Projects/Experimentation]
        [Python interpreter: /Users/cohlem/anaconda3/bin/python]
        [Exit code: 0]

# Complete Bash Code

In [69]:
from typing import Literal, Tuple

HARD_TIMEOUT = 300

PS1_BLOCK_BEGIN = '\n###PS1JSON###\n'
PS1_BLOCK_END   = '\n###PS1END###'

PS1_BLOCK_REGEX = re.compile(
    f'^{PS1_BLOCK_BEGIN.strip()}(.*?){PS1_BLOCK_END.strip()}',
    re.DOTALL | re.MULTILINE,
)

# matches blocks like these

# {
#   "pid": "37675",
#   "exit_code": "0",
#   "username": "cohlem",
#   "hostname": "Manishs-MacBook-Air",
#   "working_dir": "/Users/cohlem/Projects/Experimentation",
#   "py_interpreter_path": "/Users/cohlem/anaconda3/bin/python"
# }





class CmdOutputMetadata(BaseModel):
    exit_code: str = "-1"
    pid: str = -1
    username: str | None = None
    hostname: str | None = None
    working_dir: str | None = None
    py_interpreter_path: str | None = None

    @classmethod
    def build_ps1_prompt(cls) -> str:
        json_template = json.dumps(
            {
                'pid': '$!',
                'exit_code': '$?',
                'username': r'\u',
                'hostname': r'\h',
                'working_dir': r'$(pwd)',
                'py_interpreter_path': r'$(which python 2>/dev/null || echo "")',
            },
            indent=2,
        ).replace('"', r'\"')

        return PS1_BLOCK_BEGIN + json_template + PS1_BLOCK_END + '\n'

    @classmethod
    def parse_ps1_blocks(cls, pane_text: str) -> tuple[list[re.Match], list[dict]]:
        matches = []
        metadata = []

        for match in PS1_BLOCK_REGEX.finditer(pane_text):
            
            try:
                parsed = json.loads(match.group(1).strip())
                
                matches.append(match)
                metadata.append(parsed)
            except json.JSONDecodeError:
                print(
                    f'Could not parse PS1 block, skipping:\n{match.group(1)}\n'
                    + traceback.format_exc()
                )

        return matches, metadata


class CmdOutputObservation(BaseModel):
    content: str
    metadata: CmdOutputMetadata

    def to_agent_observation(self) -> str:
        parts = [f'<RESULT>{self.content}</RESULT>']

        if self.metadata.working_dir:
            parts.append(f'[Current working directory: {self.metadata.working_dir}]')
        if self.metadata.py_interpreter_path:
            parts.append(f'[Python interpreter: {self.metadata.py_interpreter_path}]')
        if self.metadata.exit_code:
            parts.append(f'[Exit code: {self.metadata.exit_code}]')

        return '\n'.join(parts)


class BashSession:
    PS1 = CmdOutputMetadata.build_ps1_prompt()
    HISTORY_LIMIT = 10_000

    def __init__(self, work_dir: str, username: str):
        self.work_dir = work_dir
        self.username = username
        self.server = libtmux.Server()

    def initialize(self):
        self.session = self.server.new_session(
            session_name='bash_session',
            start_directory=self.work_dir,
            kill_session=True,
            x=1000, y=1000,
        )
        self.session.set_option('history-limit', str(self.HISTORY_LIMIT))

        initial_window = self.session.active_window
        self.window = self.session.new_window(
            window_name='bash',
            window_shell='/bin/bash',
            start_directory=self.work_dir,
        )
        self.pane = self.window.active_pane
        initial_window.kill()

        self.pane.send_keys(
            f'export PROMPT_COMMAND=\'export PS1="{self.PS1}"\'; export PS2=""'
        )
        time.sleep(0.5)
        self._clear_pane()

    def _clear_pane(self) -> None:
        self.pane.send_keys('C-l', enter=False)
        time.sleep(0.1)
        self.pane.cmd('clear-history')

    def _get_pane_content(self) -> str:
        lines = self.pane.cmd('capture-pane', '-J', '-pS', '-').stdout
        return '\n'.join(line.rstrip() for line in lines)

    def _slice_output_between_markers(
        self,
        pane_content: str,
        ps1_matches: list[re.Match],
        before_last_match: bool = False,
    ) -> Tuple[str, list[str]]:
        if len(ps1_matches) == 0:
            return pane_content, [pane_content]

        if len(ps1_matches) == 1:
            if before_last_match:
                text = pane_content[: ps1_matches[0].start()]
            else:
                text = pane_content[ps1_matches[0].end() + 1 :]
            return text, [text]

        segments = []
        for i in range(len(ps1_matches) - 1):
            segment = pane_content[
                ps1_matches[i].end() + 1 : ps1_matches[i + 1].start()
            ]
            segments.append(segment)

        combined = '\n'.join(segments)
        return combined, segments

    def execute(self, command: str) -> CmdOutputObservation:
        init_content = self._get_pane_content()
        init_matches, _ = CmdOutputMetadata.parse_ps1_blocks(init_content)
        init_marker_count = len(init_matches)

        self.pane.send_keys(command) # ls -lah
        time.sleep(0.5)

        start_time = time.time()
        while time.time() - start_time < HARD_TIMEOUT:
            pane_text = self._get_pane_content()
            ps1_matches, ps1_jsons = CmdOutputMetadata.parse_ps1_blocks(pane_text)

            if len(ps1_matches) == init_marker_count + 1:
                _, segments = self._slice_output_between_markers(pane_text, ps1_matches)
                output = ''.join(segments)

                return CmdOutputObservation(
                    content=output,
                    metadata=CmdOutputMetadata(**ps1_jsons[-1]),
                )

            time.sleep(0.5)

        return CmdOutputObservation(
            content='Command timed out — it may still be running.',
            metadata=CmdOutputMetadata(),
        )

In [70]:
bash_session = BashSession(work_dir='/Users/cohlem/Projects/Experimentation', username='nanoclaudeagent')
bash_session.initialize()


In [71]:
out = bash_session.execute('ls videos')
print(out.to_agent_observation())

<RESULT>ls videos
comparision     day x of

</RESULT>
[Current working directory: /Users/cohlem/Projects/Experimentation]
[Python interpreter: /Users/cohlem/anaconda3/bin/python]
[Exit code: 0]


In [72]:
out = bash_session.execute('ls Papers')
print(out.content)

ls videos
comparision     day x of

ls Papers
RL




In [73]:
print(out.to_agent_observation())

<RESULT>ls videos
comparision     day x of

ls Papers
RL

</RESULT>
[Current working directory: /Users/cohlem/Projects/Experimentation]
[Python interpreter: /Users/cohlem/anaconda3/bin/python]
[Exit code: 0]


# Editor 


In [74]:
#imports 
import functools
import json
import os
import re
import shutil
import sys
import tempfile
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Tuple

import libtmux
from litellm import completion
from pydantic import BaseModel

## View

In [75]:


Command = Literal[
    'view',
    'create',
    'str_replace',
    'insert',
#     'undo_edit',
]
@dataclass
class FileEditObservation:
    path: Path
    success_message: str | None = None
    file_content: str | None = None
    new_file_content: str | None = None
    output: str | None = None

        
        
class Editor:
    def __init__(self):
        pass

    def __call__(
        self,
        command: Command,
        path: str,
        file_text: str | None = None,
        view_range: list[int] | None = None,
        old_str: str | None = None,
        new_str: str | None = None,
        insert_line: int | None = None,
        enable_linting: bool = False,
        **kwargs,
    ):
        _path = Path(path)
        if command == 'view':
            if view_range:
                start_line, end_line = view_range
                out = self.read_file(_path, start_line, end_line)
                success_msg = f'Successfully read the file from line {start_line} to {end_line} file content:\n{out}'
            else:
                out = self.read_file(_path)
                success_msg = f'Successfully read the file content:\n{out}'
            return FileEditObservation(
                path=_path,
                success_message=success_msg,
            )
        if command == 'create':
            self.write_file(path=_path, file_text=file_text)
            return FileEditObservation(
                path=_path,
                success_message=f'Successfully Created a file at {_path} with file content:\n{file_text}',
            )
        
        

    def read_file(self, path: Path, start_line: int | None = None, end_line: int | None = None):
        try:
            if start_line is not None and end_line is not None:
                with open(path, 'r', encoding='utf-8') as f:
                    text = []
                    for i, line in enumerate(f, 1):
                        if i >= start_line:
                            text.append(line)
                        if i > end_line:
                            break
                return ''.join(text)
            else:
                with open(path, 'r', encoding='utf-8') as f:
                    return ''.join(f)
        except Exception as e:
            raise ToolError(f'Error {e} ')


    def write_file(self, path: Path, file_text: str) -> None:
        try:
            with open(path, 'w', encoding='utf-8') as f:
                f.write(file_text)
        except Exception as e:
            raise ToolError(f'Ran into {e} while trying to write to {path}') from None

In [76]:
editor = Editor()

In [77]:
print(editor(command = 'view', path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', view_range=[25,30]).success_message)

Successfully read the file from line 25 to 30 file content:

install_requires = [
    "accelerate",
    "codetiming",
    "datasets",
    "dill",
    "hydra-core",



## Insert

In [ ]:
line 1 
line 2
NEW LINE
line 3 
line 4

In [85]:

class Editor:
    def __init__(self):
        pass

    def __call__(
        self,
        command: Command,
        path: str,
        file_text: str | None = None,
        view_range: list[int] | None = None,
        old_str: str | None = None,
        new_str: str | None = None,
        insert_line: int | None = None,
        enable_linting: bool = False,
        **kwargs,
    ):
        _path = Path(path)
        if command == 'view':
            if view_range:
                start_line, end_line = view_range
                out = self.read_file(_path, start_line, end_line)
                success_msg = f'Successfully read the file from line {start_line} to {end_line} file content:\n{out}'
            else:
                out = self.read_file(_path)
                success_msg = f'Successfully read the file content:\n{out}'
            return FileEditObservation(
                path=_path,
                success_message=success_msg,
            )
        if command == 'create':
            self.write_file(path=_path, file_text=file_text)
            return FileEditObservation(
                path=_path,
                success_message=f'Successfully Create a file at {_path} with file content:\n{file_text}',
            )
        elif command == 'insert':
            return self.insert(_path, insert_line, new_str)


    def view(self, path, view_range) -> FileEditObservation:
        start_line, end_line = view_range
        out = self.read_file(path, start_line, end_line)
        return FileEditObservation(path=path, output=out)

    def read_file(self, path: Path, start_line: int | None = None, end_line: int | None = None):
        try:
            if start_line is not None and end_line is not None:
                with open(path, 'r', encoding='utf-8') as f:
                    text = []
                    for i, line in enumerate(f, 1):
                        if i >= start_line:
                            text.append(line)
                        if i > end_line:
                            break
                return ''.join(text)
            else:
                with open(path, 'r', encoding='utf-8') as f:
                    return ''.join(f)
        except Exception as e:
            raise ToolError(f'Error {e} ')

    def insert(self, path: Path, insert_line: int, new_str: str):
        try:
            with tempfile.NamedTemporaryFile(
                mode='w', encoding='utf-8', delete=False
            ) as temp_file:
                with open(path, 'r', encoding='utf-8') as f:
                    for i, line in enumerate(f, 1):
                        if i < insert_line:
                            temp_file.write(line)
                        if i > insert_line:
                            break

                new_str_ls = new_str.split('\n')
                for line in new_str_ls:
                    temp_file.write(line + '\n')

                with open(path, 'r', encoding='utf-8') as f:
                    for i, line in enumerate(f, 1):
                        if i >= insert_line:
                            temp_file.write(line)

                shutil.move(temp_file.name, path)

            return FileEditObservation(
                success_message=f'Successfully Inserted this code {new_str}',
                path=path,
            )
        except Exception as e:
            raise ToolError(f'Error {e}')

    def write_file(self, path: Path, file_text: str) -> None:
        try:
            with open(path, 'w', encoding='utf-8') as f:
                f.write(file_text)
        except Exception as e:
            raise ToolError(f'Ran into {e} while trying to write to {path}') from None

In [86]:
editor = Editor()

In [88]:
print(editor(command = 'view', path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', view_range=[30,35]).success_message)

Successfully read the file from line 30 to 35 file content:
    "dill",
    "hydra-core",
    "numpy",
    "pandas",
    "peft",
helllo :)
    "pyarrow>=19.0.0",



In [89]:
print(editor(command = 'insert', path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', insert_line=31, new_str= 'helllo :)').success_message)

Successfully Inserted this code helllo :)


In [90]:
print(editor(command = 'view', path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', view_range=[30,35]).success_message)

Successfully read the file from line 30 to 35 file content:
    "dill",
helllo :)
    "hydra-core",
    "numpy",
    "pandas",
    "peft",
helllo :)



In [ ]:
line 1 
line 2
NEW_STR
line 3 
line 4

In [ ]:
3, OLD_STR, 13

## Str replace

In [114]:
class ToolError(Exception):
    """Raised when a tool encounters an error."""

    def __init__(self, message):
        self.message = message
        super().__init__(message)

    def __str__(self):
        return self.message
    
SNIPPET_CONTEXT_WINDOW = 4

In [91]:

class Editor:
    def __init__(self):
        pass

    def __call__(
        self,
        command: Command,
        path: str,
        file_text: str | None = None,
        view_range: list[int] | None = None,
        old_str: str | None = None,
        new_str: str | None = None,
        insert_line: int | None = None,
        enable_linting: bool = False,
        **kwargs,
    ):
        _path = Path(path)
        if command == 'view':
            if view_range:
                start_line, end_line = view_range
                out = self.read_file(_path, start_line, end_line)
                success_msg = f'Successfully read the file from line {start_line} to {end_line} file content:\n{out}'
            else:
                out = self.read_file(_path)
                success_msg = f'Successfully read the file content:\n{out}'
            return FileEditObservation(
                path=_path,
                success_message=success_msg,
            )
        if command == 'create':
            self.write_file(path=_path, file_text=file_text)
            return FileEditObservation(
                path=_path,
                success_message=f'Successfully Create a file at {_path} with file content:\n{file_text}',
            )
        elif command == 'insert':
            return self.insert(_path, insert_line, new_str)
        elif command == 'str_replace':
            return self.str_replace(_path, new_str=new_str, old_str=old_str, enable_linting=False)

    def view(self, path, view_range) -> FileEditObservation:
        start_line, end_line = view_range
        out = self.read_file(path, start_line, end_line)
        return FileEditObservation(path=path, output=out)

    def read_file(self, path: Path, start_line: int | None = None, end_line: int | None = None):
        try:
            if start_line is not None and end_line is not None:
                with open(path, 'r', encoding='utf-8') as f:
                    text = []
                    for i, line in enumerate(f, 1):
                        if i >= start_line:
                            text.append(line)
                        if i > end_line:
                            break
                return ''.join(text)
            else:
                with open(path, 'r', encoding='utf-8') as f:
                    return ''.join(f)
        except Exception as e:
            raise ToolError(f'Error {e} ')

    def insert(self, path: Path, insert_line: int, new_str: str):
        try:
            with tempfile.NamedTemporaryFile(
                mode='w', encoding='utf-8', delete=False
            ) as temp_file:
                with open(path, 'r', encoding='utf-8') as f:
                    for i, line in enumerate(f, 1):
                        if i < insert_line:
                            temp_file.write(line)
                        if i > insert_line:
                            break

                new_str_ls = new_str.split('\n')
                for line in new_str_ls:
                    temp_file.write(line + '\n')

                with open(path, 'r', encoding='utf-8') as f:
                    for i, line in enumerate(f, 1):
                        if i >= insert_line:
                            temp_file.write(line)

                shutil.move(temp_file.name, path)

            return FileEditObservation(
                success_message=f'Successfully Inserted this code {new_str}',
                path=path,
            )
        except Exception as e:
            raise ToolError(f'Error {e}')

    def str_replace(
        self,
        path: Path,
        old_str: str,
        new_str: str | None,
        enable_linting: bool,
    ) -> FileEditObservation:
        new_str = new_str or ''

        file_content = self.read_file(path)

        pattern = re.escape(old_str)
        
        occurrences = [
            (
                file_content.count('\n', 0, match.start()) + 1,
                match.group(),
                match.start(),
            )
            for match in re.finditer(pattern, file_content)
        ]

        if not occurrences:
            raise ToolError(
                f'No replacement was performed, old_str `{old_str}` did not appear verbatim in {path}.'
            )
        if len(occurrences) > 1:
            line_numbers = sorted(set(line for line, _, _ in occurrences))
            raise ToolError(
                f'No replacement was performed. Multiple occurrences of old_str `{old_str}` in lines {line_numbers}. Please ensure it is unique.'
            )

        replacement_line, matched_text, idx = occurrences[0]

        new_file_content = (
            file_content[:idx] + new_str + file_content[idx + len(matched_text):]
        )
        


        self.write_file(path, new_file_content)

        start_line = max(0, replacement_line - SNIPPET_CONTEXT_WINDOW)
        end_line = replacement_line + SNIPPET_CONTEXT_WINDOW + new_str.count('\n')

        snippet = self.read_file(path, start_line=start_line + 1, end_line=end_line)

        success_message = f'The file {path} has been edited. '
        success_message += f'a snippet of {path}\nsnippet:\n{snippet}'

        return FileEditObservation(
            success_message=success_message,
            path=path,
            file_content=file_content,
            new_file_content=new_file_content,
        )

    def write_file(self, path: Path, file_text: str) -> None:
        try:
            with open(path, 'w', encoding='utf-8') as f:
                f.write(file_text)
        except Exception as e:
            raise ToolError(f'Ran into {e} while trying to write to {path}') from None

In [92]:
editor = Editor()

In [119]:
print(editor(command = 'view', path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', view_range=[65,66]).success_message)

Successfully read the file from line 65 to 66 file content:
this_directory = Path(__file__).parent
long_description = (this_directory / "README.md").read_text()




In [120]:
print(editor(command = 'str_replace', 
             path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', 
             old_str="""this_directory = Path(__file__).parent
long_description = (this_directory / "README.md").read_text()""", 
             
             new_str="NOTHINGGGGGGG").success_message)

The file /Users/cohlem/Projects/Experimentation/verl/setup.py has been edited. a snippet of /Users/cohlem/Projects/Experimentation/verl/setup.py
snippet:
}


NOTHINGGGGGGG

setup(
    name="verl",
    version=__version__,
    package_dir={"": "."},



In [121]:
print(editor(command = 'view', path = '/Users/cohlem/Projects/Experimentation/verl/setup.py', view_range=[65,66]).success_message)

Successfully read the file from line 65 to 66 file content:
NOTHINGGGGGGG

setup(

